# Step 6C: Phenotyping with Thresholding

This notebook performs cell phenotyping using a deterministic, rule-based **thresholding** approach. For each marker, a threshold is applied to binarize its expression (positive/negative). Phenotypes are then assigned based on logical combinations of these binary marker states (e.g., a cell is a "Helper T-Cell" if it is positive for CD45, CD3e, AND CD4).

This method is highly interpretable and reproducible, serving as a valuable complement to unsupervised clustering methods.

## Key Objectives:
- **Apply Thresholds**: For each sample (or ROI), apply pre-defined thresholds to the raw marker expression data to determine which cells are "positive" for each marker.
- **Assign Phenotypes**: Use a dictionary of phenotype definitions (e.g., `{'Treg': ['CD45', 'CD3e', 'CD4', 'FoxP3']}`) to assign an initial phenotype to each cell.
- **Resolve Conflicts**:
    1. **Hierarchy**: Apply a pre-defined hierarchy to resolve conflicts between related cell types (e.g., if a cell is positive for both "Treg" and "Helper T", the more specific "Treg" label is kept).
    2. **Tie-Breaking**: For conflicts between unrelated types, the phenotype associated with the marker with the highest normalized expression wins.
- **Generate Outputs**: Add a `phenotype_threshold` column to the main dataset.

## Prerequisites:
- **Batch Corrected Data (Step 5)**: This notebook uses the **raw** marker columns from the batch-corrected file for applying thresholds but uses the **normalized** columns for tie-breaking.
- **Threshold Files**: Requires CSV files containing marker thresholds for each sample/ROI, typically generated from manual gating in a tool like FCS Express or FlowJo.

**Where thresholds come from**: either exported per-batch CSVs (`<batch>.ome.tiff_-_Merged_Image.csv`, a `Marker` column plus one or more `Polygon_*` columns whose median is used), or regenerate them from the images with `Step2B_background_detection.ipynb` / **Step 2 > Detect background from the image**.

In [ ]:
# Imports & Environment Setup
import os, sys, importlib, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# Add src to path for imports
project_root = Path().resolve().parent
src_path = project_root / 'src'
sys.path.insert(0, str(src_path))

from spacemap.config import default_config

from spacemap import ph_thres_utils
importlib.reload(ph_thres_utils)
from spacemap.ph_thres_utils import (
    apply_thresholds_per_batch,
    apply_z1_positivity,
    assign_phenotypes_binary_tiebreak,
    apply_exclusion_markers,
    apply_threshold_phenotypes,
    generate_threshold_heatmap,
)

print(f"Project root: {project_root}")
print("Step 6C: Threshold Phenotyping Setup Complete")

## Configuration

Configure the threshold-based phenotyping parameters below. These settings control the thresholding approach, normalization, and phenotype definitions.

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# I/O paths
INPUT_DIR = project_root / 'outputs/batch_corrected'
INPUT_FILE = INPUT_DIR / 'all_samples_combined_corrected.csv'
OUTPUT_DIR = default_config.output_dir / 'phenotyping'

# Threshold ROI directory (polygon gate CSVs)
THRESHOLD_ROI_DIR = project_root / 'data' / 'thresholds'

# Cache for intermediates (binary + normalized marker matrices, assignments, heatmap)
CACHE_DIR = OUTPUT_DIR / 'thr'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Column Configuration
CELL_ID_COLUMN = 'cell_id'  # Name of the cell ID column (configurable)
BATCH_COL = 'batch_id'
FILTER_COLUMN = 'corrected'
FILTER_VALUE = 1  # keep only tissue==1
ENABLE_FILTERING = True

# Threshold parameters
APPLY_THRESHOLDS = True  # Set to False to skip thresholding and use raw markers as-is
# Z1 score positivity, for projects without threshold tables (the Resolve tab's
# "Z1 score (no threshold files)" option). A marker is positive when its
# ArcSinh -> per-batch Z1 score is above Z1_CUTOFF. Z2, CDF and NLog are not
# used in this mode, because they would change what the cutoff means.
USE_Z1_POSITIVITY = False
Z1_CUTOFF = 1.5
THRESHOLD_STATISTIC = 'median'  # options include 'percentile_05' (Q0.05)
TIE_SCORE_TRANSFORM = 'asinh_difference'  # 'identity_residual', 'asinh_difference', or 'log1p_difference'
TIE_SCORE_COFACTOR = 5.0  # used only for ArcSinh
METHOD_TAG = 'threshold'
PHENOTYPE_COL = 'phenotype_threshold'  # Name of the phenotype column to add

# Normalization flags (ordered sequence for tie-breaking)
NORM_APPLY_Z1_MARKERWISE = True
NORM_APPLY_Z2_CELLWISE = False
NORM_APPLY_CDF = False
NORM_APPLY_NLOG = False
NORMALIZATION_ORDER = ['Z1']  # runs after TIE_SCORE_TRANSFORM
NORMALIZATION_GROUP_COLS = BATCH_COL   # If you have a specific column you can assign as ['File Name']

# Marker panel (raw)
MARKERS_BASE = [
    "CD31", "E-cadherin", "CD20", "CD4", "Pan-Cytokeratin", "CD68", "Podoplanin", "CD11c", "CD8", "HLA-DR",
    "CD3e", "CD45", "CD15", "FoxP3", "PDGFRa", "aSMA", "CD163", "FN1", "S100A9", "CD11b", "CD38",
    "Vimentin"
]


PHENOTYPE_DEFINITIONS = {
    'Helper T': [["CD45", "CD3", "CD4"], ["CD3", "CD4"], ["CD45", "CD4"]],
    'Treg': [["CD45", "CD3e", "CD4", "FoxP3"], ["CD3e", "CD4", "FoxP3"], ["CD45", "CD4", "FoxP3"]],
    'Cytotoxic T': [["CD45", "CD3", "CD8"], ["CD3", "CD8"], ["CD45", "CD8"]],
    'B cell': [["CD45", "CD20"]],
    'Plasma B cell': [["CD45", "CD38"]],
    'Granulocyte': [["CD45", "CD11b", "CD15"], ["CD45", "CD11b", "CD15", "S100A9"]],
    'Mac': [["CD45", "CD68", "CD163"], ["CD45","CD163"], ["CD45", "CD68"],["CD163", "CD68"],  ["HLA-DR", "CD68"], ["HLA-DR", "CD163"]],  # Monocytes often CD11b+ HLA-DR+ (vs. mac CD68+)
    'DC': [["CD45", "CD11c"], ["CD45", "CD11c", "HLA-DR"]],
    'Epithelia': [["E-cadherin"], ["Pan-Cytokeratin"]],
    'Endothelia': [["CD31"]],
    'CAF': [["FN1"], ["PDGFRa"], ["Vimentin"], ["aSMA"], ["Podoplanin"]],
}


# Hierarchy: child -> parent (parent removed if child positive)
# Biologically refined: Subtypes override (e.g., Tregs > CD4+ subtypes > CD4+ > CD3+ > immune).
# Mac/DC/monocyte under immune; structural flat but smooth > stroma.
# From top markers: Emphasize functional subtypes (e.g., GATA3+ as Th2 child of CD4+); no hierarchy for derived/rare (dirt/undefined omitted).
PHENOTYPE_HIERARCHY = {
    'Treg': 'Helper T', 'Helper T': None, 'Cytotoxic T': None,
    'B cell': None, 'Plasma B cell': None, 'Granulocyte': None,
    'Mac': None, 'DC': None, 'Epithelia': None, 'Endothelia': None, 'CAF': None,
}

print("Configuration loaded:")
print(f"  Input file: {INPUT_FILE}")
print(f"  Output dir: {OUTPUT_DIR}")
print(f"  Cell ID column: {CELL_ID_COLUMN}")
print(f"  Batch column: {BATCH_COL}")
print(f"  Phenotype column: {PHENOTYPE_COL}")
print(f"  Threshold ROI dir: {THRESHOLD_ROI_DIR}")
print(f"  Apply thresholds: {APPLY_THRESHOLDS}")
print(f"  Z1 score positivity: {USE_Z1_POSITIVITY} (cutoff {Z1_CUTOFF})")
print(f"  Threshold statistic: {THRESHOLD_STATISTIC}")
print(f"  Markers: {len(MARKERS_BASE)}")
print(f"  Phenotypes: {len(PHENOTYPE_DEFINITIONS)}")
print(f"  Normalization flags: Z1={NORM_APPLY_Z1_MARKERWISE}, Z2={NORM_APPLY_Z2_CELLWISE}, CDF={NORM_APPLY_CDF}, -Log={NORM_APPLY_NLOG}")


# Exclusion markers: these never define a phenotype, they overturn one.
# A call is only overturned when a forbidden marker is STRONGER than the
# markers that justified it. Spillover from a touching neighbour puts a little
# of almost every marker into almost every cell - roughly one in ten true CD8
# cells carries some CD4 - so vetoing on 'any signal at all' discards far more
# correct calls than wrong ones (measured: CD8 F1 0.865 -> 0.828). The relative
# test moves a tenth as many cells and raises CD8 F1 to 0.876 instead.
PHENOTYPE_EXCLUSIONS = {
    'CD8':        ['CD4', 'FoxP3'],
    'Helper T':   ['CD8', 'FoxP3'],
    'Treg':       ['CD8'],
    'B cell':     ['CD3e'],
    'Plasma B cell': ['CD3e'],
    'DC':         ['CD3e', 'CD20'],
    'Mac':        ['CD3e', 'CD20'],
    'Epithelia':  ['CD45'],
    'Endothelia': ['CD45'],
    'CAF':        ['CD45'],   # highest-value rule: fixes CD4/Treg/CD8 -> CAF errors
}
REASSIGN_VETOED = True   # False sends vetoed cells to the unclassified bucket instead
print(f'Exclusion rules: {len(PHENOTYPE_EXCLUSIONS)} phenotypes')

In [ ]:
# =============================================================================
# STEP 1: DATA LOADING AND PREPARATION
# =============================================================================
print("Step 1: Loading data and preparing threshold input...")

if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing input file: {INPUT_FILE}')

print(f'Loading: {INPUT_FILE}')
df = pd.read_csv(INPUT_FILE, low_memory=False)
original_count = len(df)
print(f'Loaded: {original_count:,} rows')

# Apply flexible filtering criteria
if ENABLE_FILTERING and FILTER_COLUMN in df.columns:
    before = len(df)
    df = df[df[FILTER_COLUMN] == FILTER_VALUE].copy()
    print(f'Filter {FILTER_COLUMN}=={FILTER_VALUE}: kept {len(df):,} / {before:,}')

# Verify the configurable cell ID column exists
if CELL_ID_COLUMN not in df.columns:
    raise ValueError(f"Critical: '{CELL_ID_COLUMN}' column missing from input data")

print(f"Cell ID validation: {df[CELL_ID_COLUMN].nunique():,} unique cell_ids out of {len(df):,} rows")

# Verify required markers
missing = [m for m in MARKERS_BASE if m not in df.columns]
if missing:
    raise ValueError(f'Missing required marker columns: {missing}')

print(f'Unique batches: {df[BATCH_COL].nunique()}')
print(f'Markers available: {len(MARKERS_BASE)}')

# Keep original dataframe intact to preserve spatial coordinates and all columns
original_df = df.copy()

print(f"\nStep 1 Summary:")
print(f" Original rows: {original_count:,}")
print(f" After filtering: {len(df):,}")
print(f" Cell ID column: {CELL_ID_COLUMN} with {df[CELL_ID_COLUMN].nunique():,} unique values")
print(f" Ready for threshold-based phenotyping")

## Step 2: Threshold Processing

Load and aggregate polygon thresholds for each batch/ROI, then apply threshold subtraction to raw markers.

In [ ]:
# =============================================================================
# STEP 2: THRESHOLD PROCESSING
# =============================================================================
print('Step 2: Applying batch-specific polygon thresholds...')
import time
start_time = time.time()

if USE_Z1_POSITIVITY:
    binary_df, norm_df = apply_z1_positivity(
        df,
        MARKERS_BASE,
        batch_col=BATCH_COL,
        method_tag=METHOD_TAG,
        cutoff=Z1_CUTOFF,
        cofactor=TIE_SCORE_COFACTOR,
    )
    print(f"Z1 score positivity: positive where Z1 > {Z1_CUTOFF}")
else:
    binary_df, norm_df = apply_thresholds_per_batch(
        df_main=df,
        method_tag=METHOD_TAG,
        threshold_roi_dir=THRESHOLD_ROI_DIR,
        markers_for_thresholding_raw=MARKERS_BASE,
        threshold_statistic=THRESHOLD_STATISTIC,
        norm_group_cols=NORMALIZATION_GROUP_COLS,
        norm_flags=(
            NORM_APPLY_Z1_MARKERWISE,
            NORM_APPLY_Z2_CELLWISE,
            NORM_APPLY_CDF,
            NORM_APPLY_NLOG,
        ),
        norm_order=NORMALIZATION_ORDER,
        batch_col=BATCH_COL,
        apply_thresholds=APPLY_THRESHOLDS,
        tie_score_transform=TIE_SCORE_TRANSFORM,
        tie_score_cofactor=TIE_SCORE_COFACTOR,
    )

# Optionally persist intermediates
(binary_df.astype('int16') if len(binary_df) else binary_df).to_parquet(CACHE_DIR / 'binary_markers.parquet')
norm_df.to_parquet(CACHE_DIR / 'normalized_markers.parquet')

processing_time = time.time() - start_time
print(f"\nStep 2 Summary:")
print(f" Binary marker columns: {len(binary_df.columns)}")
print(f" Normalized marker columns: {len(norm_df.columns)}")
print(f" Processing time: {processing_time:.1f}s")
print(f" Intermediate files saved to: {CACHE_DIR}")

## Step 3: Phenotype Assignment

Build phenotype boolean matrix and resolve conflicts using normalized marker values.

In [ ]:
# =============================================================================
# STEP 3: PHENOTYPE ASSIGNMENT AND CONFLICT RESOLUTION
# =============================================================================
print('Step 3: Assigning phenotypes...')
start_time = time.time()

bin_cols = binary_df.columns.tolist()
norm_cols = norm_df.columns.tolist()

assignment_df = assign_phenotypes_binary_tiebreak(
    df_processed=pd.concat([binary_df, norm_df], axis=1),
    bin_marker_cols=bin_cols,
    norm_marker_cols=norm_cols,
    phenotype_col_name=PHENOTYPE_COL,
    phenotype_definitions=PHENOTYPE_DEFINITIONS,
    phenotype_hierarchy=PHENOTYPE_HIERARCHY,
)

# --- Exclusion markers -------------------------------------------------
if PHENOTYPE_EXCLUSIONS:
    before = assignment_df[PHENOTYPE_COL].copy()
    assignment_df[PHENOTYPE_COL] = apply_exclusion_markers(
        assignment_df[PHENOTYPE_COL],
        pd.concat([binary_df, norm_df], axis=1),
        norm_cols,
        PHENOTYPE_DEFINITIONS,
        PHENOTYPE_EXCLUSIONS,
        reassign=REASSIGN_VETOED,
    )
    moved = int((before != assignment_df[PHENOTYPE_COL]).sum())
    print(f'Exclusion markers moved {moved:,} cells ({moved/len(assignment_df):.2%})')
    if moved:
        shifts = pd.crosstab(before[before != assignment_df[PHENOTYPE_COL]],
                             assignment_df[PHENOTYPE_COL][before != assignment_df[PHENOTYPE_COL]])
        display(shifts)

# Save phenotype assignments
assignment_df.to_parquet(CACHE_DIR / 'phenotype_assignment.parquet')

assignment_time = time.time() - start_time
print(f"\nStep 3 Summary:")
print(f" Phenotype assignment completed in {assignment_time:.1f}s")
print(f" Assignments saved to: {CACHE_DIR / 'phenotype_assignment.parquet'}")

print('\nPhenotype distribution:')
assignment_counts = assignment_df[PHENOTYPE_COL].value_counts()
for ph, cnt in assignment_counts.head(10).items():
    print(f'  {ph}: {cnt:,} ({cnt/len(assignment_df):.1%})')
if len(assignment_counts) > 10:
    print(f'  ... and {len(assignment_counts) - 10} more phenotypes')

## Step 4: Expression Heatmap Generation

Generate heatmap visualization of mean normalized marker expression per phenotype.

In [ ]:
# =============================================================================
# STEP 4: EXPRESSION HEATMAP GENERATION
# =============================================================================
print('Step 4: Generating expression heatmap...')

# Prepare heatmap input data
heatmap_input = norm_df.copy()
heatmap_input[PHENOTYPE_COL] = assignment_df[PHENOTYPE_COL]

# Determine suffix automatically from normalized columns
norm_suffix_detect = '_norm_' + METHOD_TAG

# Generate heatmap
ax = generate_threshold_heatmap(
    df=heatmap_input,
    assignment_col=PHENOTYPE_COL,
    markers=MARKERS_BASE,
    marker_suffix=norm_suffix_detect,
    figsize=(12, max(6, 0.5 * len(PHENOTYPE_DEFINITIONS))),
    save_path=CACHE_DIR / 'phenotype_threshold_heatmap.png'
)

plt.title('Threshold Phenotyping: Normalized Marker Profile', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Heatmap saved to: {CACHE_DIR / 'phenotype_threshold_heatmap.png'}")

## Step 5: Apply Phenotypes and Save Results

Apply the phenotype assignments to the dataset and save final results with proper data preservation.

In [ ]:
# =============================================================================
# STEP 5: APPLY PHENOTYPES AND SAVE RESULTS
# =============================================================================
print("Step 5: Applying phenotypes and saving results...")

# FIXED: Use proper apply function instead of anchor_and_update_unified
# This ensures the full dataset is preserved and properly merged by cell_id
clean_data = original_df[[c for c in original_df.columns if not c.endswith('_norm')]].copy()
result = apply_threshold_phenotypes(
    data=clean_data,
    assignment_df=assignment_df,
    phenotype_col=PHENOTYPE_COL,
    output_base_dir=OUTPUT_DIR,
    method_prefix=METHOD_TAG,
    cell_id_col=CELL_ID_COLUMN
)

print(f"\nStep 5 Results:")
print(f" Unified file: {result['unified_file']}")
print(f" Summary file: {result['summary_file']}")
print(f" Mapping file: {result['mapping_file']}")

print(f"\nPhenotype Summary:")
summary_display = result['summary_df'].head(10)
for _, row in summary_display.iterrows():
    phenotype = row.name
    count = int(row['cell_count'])
    pct = row['percentage']
    print(f" {phenotype}: {count:,} cells ({pct:.1f}%)")

if len(result['summary_df']) > 10:
    print(f" ... and {len(result['summary_df']) - 10} more phenotypes")

# Validation: Check unified file integrity
unified_df = result['data']
print(f"\nData Integrity Check:")
print(f" Original cells: {len(original_df):,}")
print(f" Unified output cells: {len(unified_df):,}")
print(f" Cell ID preservation: {unified_df[CELL_ID_COLUMN].nunique():,} unique IDs")

if PHENOTYPE_COL in unified_df.columns:
    print(f" Threshold phenotype column added successfully")
    unassigned = unified_df[PHENOTYPE_COL].isnull().sum()
    if unassigned > 0:
        print(f" Warning: {unassigned} cells have no phenotype assignment")
    else:
        print(f" All cells successfully assigned phenotypes")
else:
    print(" Warning: phenotype_threshold column not found in output")
    
print(f"\nThreshold phenotyping complete. Results saved to: {result['unified_file']}")

## Step 6: Optional Batch Export

Export individual batch files for downstream analysis or validation.

In [ ]:
# =============================================================================
# STEP 6: OPTIONAL BATCH EXPORT
# =============================================================================
import re

# Export configuration
EXPORT_ALL_BATCHES = False          # Set True to export every batch_id
EXPORT_PREFIXES = ['A1']     # List of prefixes to match (ignored if EXPORT_ALL_BATCHES=True)
EXPORT_OUTPUT_SUBDIR = CACHE_DIR / 'batch_exports'
EXPORT_OUTPUT_SUBDIR.mkdir(parents=True, exist_ok=True)

print("Step 6: Optional batch export...")

# Load unified file if not already in memory
if 'result' in globals() and isinstance(result, dict) and 'unified_file' in result:
    unified_path = result['unified_file']
    unified_df = result['data']
else:
    unified_path = OUTPUT_DIR / 'all_samples_combined_corrected_phenotyped.csv'
    if not unified_path.exists():
        raise RuntimeError(f'Cannot load unified file for export: {unified_path}')
    unified_df = pd.read_csv(unified_path)

if BATCH_COL not in unified_df.columns:
    print(f"Column '{BATCH_COL}' not found; skipping export.")
else:
    batches = unified_df[BATCH_COL].dropna().astype(str).unique().tolist()
    if EXPORT_ALL_BATCHES:
        target_batches = batches
        print(f'Exporting ALL {len(target_batches)} batches...')
    else:
        # Collect batches whose name starts with any provided prefix
        target_batches = []
        for prefix in EXPORT_PREFIXES:
            matched = [b for b in batches if b.startswith(prefix)]
            target_batches.extend(matched)
        # De-duplicate while preserving order
        seen = set()
        target_batches = [b for b in target_batches if not (b in seen or seen.add(b))]
        print(f'Exporting {len(target_batches)} batches matching prefixes {EXPORT_PREFIXES}')
    
    if not target_batches:
        print('No batches match the export criteria.')
    else:
        for b in target_batches:
            df_b = unified_df[unified_df[BATCH_COL].astype(str) == b].copy()
            safe_name = re.sub(r'[\\/*?:"<>|\s]+', '_', b)
            out_path = EXPORT_OUTPUT_SUBDIR / f'{safe_name}_{PHENOTYPE_COL}.csv'
            df_b.to_csv(out_path, index=False)
            print(f" Exported {b}: {len(df_b):,} cells → {out_path.name}")
        print(f'\nExport complete. Files saved under: {EXPORT_OUTPUT_SUBDIR}')

print("Step 6 complete.")

**Export Configuration**:
- Set `EXPORT_ALL_BATCHES = True` to export all batches
- Set `EXPORT_ALL_BATCHES = False` and specify `EXPORT_PREFIXES` for selective export
- Files are saved to `phenotyping/thr/batch_exports/` directory

This step is optional and doesn't affect the main unified output.

## Step 7: Cleanup Normalized Cache

Delete any cached normalized data to ensure fresh computation on next run. This prevents cache inconsistencies and ensures reproducibility.

In [ ]:
# =============================================================================
# STEP 7: CLEANUP NORMALIZED CACHE
# =============================================================================
print("Step 7: Cleaning up normalized cache...")

# Delete any cached files to ensure fresh computation on next run
cache_files = [
    CACHE_DIR / 'binary_markers.parquet',
    CACHE_DIR / 'normalized_markers.parquet', 
    CACHE_DIR / 'phenotype_assignment.parquet'
]

cleaned_count = 0
for cache_file in cache_files:
    if cache_file.exists():
        cache_file.unlink()
        print(f"Deleted cache: {cache_file.name}")
        cleaned_count += 1

# Also clean up the cache directory if it's empty (except for plots)
if CACHE_DIR.exists():
    remaining_files = [f for f in CACHE_DIR.iterdir() if f.suffix not in ['.png', '.jpg', '.pdf']]
    if not remaining_files:
        # Only remove non-plot files, keep visualization files
        print(f"Cache directory cleaned (keeping visualization files)")
    else:
        print(f"Cache directory contains {len(remaining_files)} remaining files")

if cleaned_count > 0:
    print(f"Cache cleanup complete. Deleted {cleaned_count} cache files.")
    print("Fresh computation will be performed on next run.")
else:
    print("No cache files found to clean up.")

## Summary

This notebook performs threshold-based phenotyping with a professional, publication-ready structure identical to the Leiden and SOM clustering methods.

### Workflow Steps
1. **Data Loading** - Load batch-corrected data with flexible filtering
2. **Threshold Processing** - Apply batch-specific polygon thresholds and normalization
3. **Phenotype Assignment** - Binary classification with hierarchy and conflict resolution
4. **Expression Heatmap** - Visualization of phenotype marker profiles
5. **Apply Phenotypes** - Generate unified output with preserved data integrity
6. **Batch Export** - Optional individual batch file generation
7. **Cache Cleanup** - Remove temporary files for fresh computation on next run

### Key Outputs
**Main Results** (stored in `phenotyping/`)
- `all_samples_combined_corrected_phenotyped.csv` - Unified dataset with threshold phenotypes
- `threshold_phenotype_summary.csv` - Phenotype statistics and cell counts
- `threshold_phenotype_mapping.csv` - Phenotype to cell count mapping

**Visualizations** (stored in `phenotyping/thr/`)
- Phenotype expression heatmaps
- Binary and normalized marker intermediate files

### Key Features
- **ROI-specific thresholding** - Batch-specific polygon threshold calculation
- **Deterministic approach** - Rule-based phenotyping for high interpretability
- **Flexible normalization** - Multi-step normalization for robust tie-breaking
- **Hierarchical resolution** - Parent-child phenotype conflict handling
- **Data integrity** - Preserves spatial coordinates and cell ID anchoring
- **Cache management** - Automatic cleanup ensures fresh computation on next run

### Method Highlights
- Median polygon threshold subtraction per batch/ROI
- OR-of-AND logic for flexible phenotype definitions
- Normalized marker intensity for unrelated conflict resolution
- Compatible output format with other phenotyping methods
- Professional visualization with BWR colormap matching other methods

### Next Steps
The unified output file is ready for downstream spatial analysis and can be combined with results from other phenotyping methods (Leiden, SOM, SCIMAP) for comprehensive validation and analysis.

## Exclusion markers - what they do

The rules above say what a phenotype *must* have. Exclusion markers say what would
disprove it, and they are applied last, after the hierarchy and tie-break.

The test is **relative, not absolute**. A cell keeps its call unless a forbidden marker is
stronger than the markers that justified it:

| cell | CD8 | CD4 | outcome |
|---|---|---|---|
| clean CD8 T cell | 0.90 | 0.00 | stays **CD8** |
| CD8 T cell touching a CD4 T cell | 0.85 | 0.12 | stays **CD8** - faint spill is ignored |
| actually a CD4 T cell | 0.20 | 0.75 | becomes **CD4** |

Measured on our reference cells: an absolute veto (any CD4 signal at all) moved 960 cells and
*lowered* CD8 F1 from 0.865 to 0.828. The relative veto moves 253 cells and raises it to 0.876.
Overall macro-F1 goes from 0.786 to about 0.792-0.795 once the CD45 rules are included.